In [1]:
# ── Data handling ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import re

# ── Lyric scraping ────────────────────────────────────────────────────────────
import lyricsgenius                     # Genius API wrapper for pulling song lyrics

# Functions

In [2]:
# ── Functions for Song Searching and Processing ────────────────────────────────────────────────
def clean_lyrics(raw_lyrics):
    """
    Strips metadata and standardizes raw Genius lyrics into clean plain text.
    This function removes that metadata along with section headers, extra
    whitespace, punctuation, and capitalisation.

    Args:
        raw_lyrics (str): Raw lyrics string returned by genius.search_song()

    Returns:
        str: Cleaned, lowercased lyrics ready for tokenization
    """
    
    # Genius prepends metadata before the first section header e.g. [Intro]
    starter = ']' # Finding the first ']' and slicing from there drops all prepended metadata
    position = raw_lyrics.find(starter)
    lyrics = raw_lyrics[position+1:]

    # Remove section headers like [Chorus], [Verse 1], etc.
    lyrics = re.sub(r"\[.*?\]", "", lyrics)

    # Remove extra whitespace, newlines
    lyrics = re.sub(r"\n+", " ", lyrics)
    lyrics = re.sub(r"\s+", " ", lyrics)

    # Remove punctuation
    lyrics = re.sub(r"[^\w\s']", "", lyrics)

    # Lowercase
    lyrics = lyrics.lower()

    return lyrics


def song_search(song,artist):
    """
    Fetches and cleans lyrics for a given song from the Genius API.

    Initialises a Genius API client, searches for the song by name and
    artist, then passes the raw lyrics through clean_lyrics() before
    returning them.

    Args:
        song (str):   Title of the song to search for
        artist (str): Name of the artist who recorded the song

    Returns:
        str: Cleaned lyrics string
    """

    genius = lyricsgenius.Genius(genius_token, timeout=60)
    song = genius.search_song(song, artist)
    lyrics = song.lyrics

    # Handles if song is not found
    if song is None:
        print(f"Could not find lyrics for '{song}' by {artist}")
        return None
    
    cleaned_lyrics = clean_lyrics(lyrics)
    return cleaned_lyrics

In [3]:
df = pd.read_csv(r'C:\Users\Monado\Documents\Data_git\Projects\Lyrics_NLP\english_song_dataset.csv')
print(df)

                   song             artist
0              Hey Jude        The Beatles
1             Let It Be        The Beatles
2             Yesterday        The Beatles
3         Come Together        The Beatles
4    A Hard Day's Night        The Beatles
..                  ...                ...
905            Gasoline               HAIM
906            Lazy Eye  Silversun Pickups
907        Panic Switch  Silversun Pickups
908          Nightlight           ILLENIUM
909          Paper Thin           ILLENIUM

[910 rows x 2 columns]


In [ ]:
# For loop for adding processed song lyrics to .txt file for storage
for i in range(0,len(df['songs'])):
    cleaned_lyrics = song_search(df['song'].iloc[i], df['artist'].iloc[i])
    with open("lyrics_list.txt", "a", encoding="utf-8") as file:
        file.write("\n" + cleaned_lyrics + ',' )